# E-Commerce Sales & Customer Analytics Walkthrough
### End-to-End Analytics Pipeline: SQL · Python · ML · Power BI

**Author:** Data Analyst / Analytics Engineer  
**Tech Stack:** SQLite, Python (`pandas`, `scikit-learn`, `matplotlib`, `seaborn`), Power BI, DAX  

---

## 1. Project Overview & Business Objectives
This project analyzes **6,500 e-commerce orders** and **12,920 order line items** spanning 2023–2025 across 800 customers and 216 SKUs.

### Key Questions Addressed:
1. **Sales Performance:** How are revenue, profit, and order volumes trending month-over-month and year-over-year?
2. **Pricing Policy:** Does aggressive discounting grow revenue, or does it compress margins into negative territory?
3. **Customer Value:** Who are our highest-value customers, and how are customer segments distributed (RFM & K-Means)?
4. **Customer Retention:** What percentage of customers return after their first purchase (Cohort Retention)?
5. **Predictive Forecasting:** What is the projected revenue for the next 6 months?
6. **Product Affinity:** Which products are frequently bought together (Market Basket Analysis)?

## 2. Environment Setup & Data Loading
We import core data science libraries and define project paths relative to the root.

In [ ]:
import os
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)

# Paths
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = BASE_DIR / 'data'
DB_PATH = DATA_DIR / 'ecommerce.db'

print('Project directory:', BASE_DIR)
print('Database exists:', DB_PATH.exists())

## 3. Data Quality & Referential Integrity QA
Before any statistical modeling, we validate the data to ensure 0 orphaned records, 0 negative sales, and chronological consistency.

In [ ]:
customers = pd.read_csv(DATA_DIR / 'dim_customers.csv')
products = pd.read_csv(DATA_DIR / 'dim_products.csv')
orders = pd.read_csv(DATA_DIR / 'fact_orders.csv', parse_dates=['order_date', 'ship_date'])
items = pd.read_csv(DATA_DIR / 'fact_order_items.csv')
returns = pd.read_csv(DATA_DIR / 'fact_returns.csv')

# Data Quality Summary
qa_summary = pd.DataFrame({
    'Table': ['dim_customers', 'dim_products', 'fact_orders', 'fact_order_items', 'fact_returns'],
    'Rows': [len(customers), len(products), len(orders), len(items), len(returns)],
    'Nulls': [customers.isna().sum().sum(), products.isna().sum().sum(), orders.isna().sum().sum(), items.isna().sum().sum(), returns.isna().sum().sum()],
    'Duplicates': [customers.duplicated().sum(), products.duplicated().sum(), orders.duplicated().sum(), items.duplicated().sum(), returns.duplicated().sum()]
})
print('Data Quality Pass:')
display(qa_summary)

# Referential Integrity Checks
orphan_orders = len(orders[~orders['customer_id'].isin(customers['customer_id'])])
orphan_items = len(items[~items['order_id'].isin(orders['order_id'])])
invalid_dates = len(orders[orders['ship_date'] < orders['order_date']])
print(f'Orphan Orders: {orphan_orders} | Orphan Items: {orphan_items} | Ship Before Order: {invalid_dates}')

## 4. Relational SQL Analytics Layer
We connect to `data/ecommerce.db` to execute business queries with window functions (`LAG`, `NTILE`, `RANK`) and CTEs.

In [ ]:
conn = sqlite3.connect(DB_PATH)

# Query 1.5: Margin Erosion by Discount Band
discount_query = '''
SELECT
    CASE
        WHEN discount = 0 THEN '0%'
        WHEN discount <= 0.10 THEN '1-10%'
        WHEN discount <= 0.20 THEN '11-20%'
        ELSE '21%+'
    END AS discount_band,
    COUNT(*) AS line_items,
    ROUND(SUM(sales), 2) AS revenue,
    ROUND(SUM(profit), 2) AS profit,
    ROUND(100.0 * SUM(profit) / NULLIF(SUM(sales), 0), 2) AS margin_pct
FROM fact_order_items
GROUP BY 1
ORDER BY 1;
'''
df_discount = pd.read_sql_query(discount_query, conn)
print('Discount Impact on Profit Margin:')
display(df_discount)

> **Key Takeaway:** Full-price orders yield a healthy **16.1% profit margin**. Modest discounts (1-10%) yield **7.1%**. However, discounts beyond **20% deeply compress margins into negative territory (-19.9%)**, causing net dollar losses.

### Top 10 Revenue-Generating SKUs

In [ ]:
top_prod_query = '''
SELECT
    p.product_name, p.category, p.sub_category,
    SUM(oi.quantity) AS units_sold,
    ROUND(SUM(oi.sales), 2) AS revenue,
    ROUND(SUM(oi.profit), 2) AS profit,
    ROUND(100.0 * SUM(oi.profit) / NULLIF(SUM(oi.sales), 0), 2) AS margin_pct
FROM fact_order_items oi
JOIN dim_products p ON p.product_id = oi.product_id
GROUP BY p.product_id
ORDER BY revenue DESC
LIMIT 10;
'''
df_top_products = pd.read_sql_query(top_prod_query, conn)
display(df_top_products)

## 5. Exploratory Data Analysis & Visualizations
Let's visualize the core drivers of sales, regional revenue distribution, and monthly seasonality.

In [ ]:
clean_df = pd.read_csv(DATA_DIR / 'clean' / 'analysis_ready.csv')

# Monthly Trend Plot
monthly = clean_df[clean_df['order_month'] <= '2025-12'].groupby('order_month').agg(
    revenue=('sales', 'sum'),
    profit=('profit', 'sum')
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(monthly['order_month'], monthly['revenue'] / 1000, color='#4C72B0', alpha=0.85, label='Revenue ($k)')
ax1.set_ylabel('Revenue ($ in Thousands)', fontsize=11)
ax1.tick_params(axis='x', rotation=90)

ax2 = ax1.twinx()
ax2.plot(monthly['order_month'], monthly['profit'] / 1000, color='#DD8452', marker='o', linewidth=2, label='Profit ($k)')
ax2.set_ylabel('Profit ($ in Thousands)', fontsize=11)
ax2.grid(False)

fig.suptitle('Monthly Revenue & Profit Performance (2023–2025)', fontsize=13, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

## 6. Customer Segmentation (RFM + K-Means Clustering)
We implement a dual segmentation framework: rule-based RFM scoring for business interpretability and unsupervised K-Means clustering for objective data validation.

In [ ]:
segments_df = pd.read_csv(DATA_DIR / 'customer_segments.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. RFM Distribution
order_rfm = segments_df['rfm_segment'].value_counts().index
sns.countplot(data=segments_df, y='rfm_segment', order=order_rfm, ax=axes[0], color='#4C72B0')
axes[0].set_title('Customer Distribution by RFM Segment', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Customers')

# 2. K-Means Scatter
sns.scatterplot(
    data=segments_df, x='frequency', y='monetary', hue='cluster_name',
    palette='deep', s=50, alpha=0.8, ax=axes[1]
)
axes[1].set_title('K-Means Clusters: Frequency vs. Total Spend', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Order Frequency')
axes[1].set_ylabel('Total Spend ($)')

plt.tight_layout()
plt.show()

## 7. Cohort Retention Decay
Tracking customer repeat engagement across cohorts based on their initial acquisition month.

In [ ]:
cohort_df = pd.read_csv(DATA_DIR / 'cohort_retention.csv', index_col='cohort_month')

plt.figure(figsize=(12, 6))
sns.heatmap(cohort_df.iloc[:, :12] * 100, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label': 'Retention %'})
plt.title('Monthly Cohort Retention Rate (%) - First 12 Months', fontsize=13, fontweight='bold')
plt.xlabel('Months Since First Acquisition')
plt.ylabel('Cohort Month')
plt.tight_layout()
plt.show()

## 8. Predictive Revenue Forecasting
Using cyclical trigonometric month encoding and Ridge regression to project revenue 6 months forward.

In [ ]:
forecast_df = pd.read_csv(DATA_DIR / 'revenue_forecast.csv')
print('6-Month Forward Revenue Forecast:')
display(forecast_df)

## 9. Market Basket & Product Affinity Analysis
Association rules derived from multi-item baskets to identify cross-sell opportunities.

In [ ]:
rules_df = pd.read_csv(DATA_DIR / 'market_basket_rules.csv')
print('Top Cross-Sell Product Associations (by Lift):')
display(rules_df.head(10))

## 10. Executive Strategic Recommendations

1. **Discontinue >20% Blanket Discounts:** Shift from deep discounting to targeted bundling. Orders with discounts >20% generate negative margins (-19.9%).
2. **VIP Retention Program for Champions & High-Value Cluster:** Top ~7% of customers account for a disproportionate share of revenue. Create exclusive loyalty benefits.
3. **Cross-Sell Product Bundles:** Pair high-lift categories (e.g., Bookcases + Storage, Machines + Supplies) directly at checkout to drive higher AOV.
4. **Intervene in Month 1 of Customer Lifecycle:** The steepest cohort churn occurs between Month 0 and Month 1 (from 100% to ~25%). An automated Day-21 post-purchase email sequence is the highest-leverage retention touchpoint.